In [1]:
# !pip install torchvision

In [2]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

# Dataset and DataLoader

In [3]:
# images load => transform => dataset of all imgs

class ImageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of all images path
        self.all_img_paths = [ os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path) ]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        # apply transformation
        if self.transformations:
            img = self.transformations(img)

        return img

In [4]:
root_dir_path = "./img_align_celeba"
for img in os.listdir(root_dir_path):
    print(os.path.join(root_dir_path, img))

./img_align_celeba\000001.jpg
./img_align_celeba\000002.jpg
./img_align_celeba\000003.jpg
./img_align_celeba\000004.jpg
./img_align_celeba\000005.jpg
./img_align_celeba\000006.jpg
./img_align_celeba\000007.jpg
./img_align_celeba\000008.jpg
./img_align_celeba\000009.jpg
./img_align_celeba\000010.jpg
./img_align_celeba\000011.jpg
./img_align_celeba\000012.jpg
./img_align_celeba\000013.jpg
./img_align_celeba\000014.jpg
./img_align_celeba\000015.jpg
./img_align_celeba\000016.jpg
./img_align_celeba\000017.jpg
./img_align_celeba\000018.jpg
./img_align_celeba\000019.jpg
./img_align_celeba\000020.jpg
./img_align_celeba\000021.jpg
./img_align_celeba\000022.jpg
./img_align_celeba\000023.jpg
./img_align_celeba\000024.jpg
./img_align_celeba\000025.jpg
./img_align_celeba\000026.jpg
./img_align_celeba\000027.jpg
./img_align_celeba\000028.jpg
./img_align_celeba\000029.jpg
./img_align_celeba\000030.jpg
./img_align_celeba\000031.jpg
./img_align_celeba\000032.jpg
./img_align_celeba\000033.jpg
./img_alig

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
transformations = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [6]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [7]:
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

# Generator Network

In [8]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [9]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3):    # 3 is for RGB
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
    
            nn.Linear(256, 512),
            nn.ReLU(),
    
            nn.Linear(512, 1024),
            nn.ReLU(),
    
            nn.Linear(1024, 64*64*img_channels),
            nn.Tanh()  # range (-1, 1)
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img


# Discriminator Network

In [10]:
class Discriminator(nn.Module):

    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()

        # fully connencted layer
        self.model = nn.Sequential(
            nn.Flatten(), #4D tensor => 1D tensor
    
            nn.Linear(64*64*img_channels, 1024),
            nn.LeakyReLU(0.2, inplace=True),
        
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
    
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
    
            nn.Linear(256, 1),
            nn.Sigmoid()  # Probability of being real/fake
        )

    def forward(self, img):
        return self.model(img)

In [11]:
GAN_Loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.002, betas=(0.5, 0.999))

# Device Selection

In [12]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"Selected Device : {device}")

Selected Device : cuda


In [13]:
generator = generator.to(device)
discriminator = discriminator.to(device)

In [14]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(next(generator.parameters()).device)
print(next(discriminator.parameters()).device)

True
NVIDIA GeForce RTX 3050 Laptop GPU
cuda:0
cuda:0


# Training the GAN

In [15]:
def train(generator, discriminator, dataloader, epochs=10):
    
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # create real and fake labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # Train the discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100)).to(device)

            real_loss = GAN_Loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_Loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss)/2

            # backpropagation for discriminator
            d_loss.backward()
            d_optimizer.step()


            # Train the Generator
            g_optimizer.zero_grad()
            
            g_loss = GAN_Loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()
            
            
            if(i%50==0):
                print(f"For Epoch : {epoch+1}/{epochs} batch: {i+1}; G-loss: {g_loss}, D-loss: {d_loss}")

        # save generated images after each epoch
        save_generated_images(generator, epoch, device)